# Error Handling - Global Error Strategies

## Purpose
Learn how to implement global error handling strategies for agent execution failures. Graceful error handling ensures your application provides helpful feedback when agents encounter limits or unexpected conditions.

## Key Concepts
- **Error Handlers**: Custom functions for handling specific error types
- **RunErrorHandlerInput**: Context information about the error
- **RunErrorHandlerResult**: Structured response with fallback output
- **Supported Error Types**: max_turns, model_refusal, invalid_final_output

## Installation

In [ ]:
#!pip install openai
#!pip install openai-agents
#!pip install aws-bedrock-token-generator

## Authentication Setup

In [ ]:
from openai import AsyncOpenAI
from agents import (
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)
from aws_bedrock_token_generator import provide_token

client = AsyncOpenAI(
    api_key=provide_token(),
    base_url="https://bedrock-mantle.us-east-1.api.aws/openai/v1",
    project="default"
)

set_default_openai_client(client)
set_default_openai_api("responses")
set_tracing_disabled(True)  # OpenAI-platform tracing can't reach Mantle

## Import Libraries

Import error handling types for creating custom handlers:

In [ ]:
import asyncio
from agents import Agent, Runner, RunErrorHandlerInput, RunErrorHandlerResult

## Step 1: Create Agent

Create a regular agent - error handling is configured at execution time:

In [ ]:
agent = Agent(
    name="Assistant",
    instructions="Be concise",
    model="openai.gpt-5.5",
)

## Step 2: Define Error Handler Function

Create a handler function for specific error types:

**Function Signature**:
- Input: `RunErrorHandlerInput[Context]` - Error details and context
- Output: `RunErrorHandlerResult` - Fallback response configuration

**RunErrorHandlerResult Fields**:
- `final_output`: Message to return to user
- `include_in_history`: Whether to add this to conversation history (default: False)

💡 **Design Tip**: Provide actionable guidance in error messages - tell users how to fix the issue.

In [ ]:
def on_max_turns(_data: RunErrorHandlerInput[None]) -> RunErrorHandlerResult:
    return RunErrorHandlerResult(
        final_output="I couldn't finish within the turn limit. Please narrow the request.",
        include_in_history=False,
    )

## Step 3: Run with Error Handlers

Pass error handlers dictionary to `Runner.run()`:

**Supported Error Types**:
- `"max_turns"` - Agent exceeded max_turns limit
- `"model_refusal"` - Model refused to respond (policy violation)
- `"invalid_final_output"` - Output doesn't match expected schema

🔍 **How It Works**: If agent hits max_turns, the handler provides a user-friendly fallback message.

🎯 **Result**: Graceful failure instead of exception!

In [ ]:
## Supported errors are - "max_turns", "model_refusal", and "invalid_final_output"

result = await Runner.run(agent, "Analyze this long transcript", max_turns=3, error_handlers={"max_turns": on_max_turns})
print(result.final_output)

## 🎉 Congratulations!

You've completed the **Error Handling** notebook!